# Enhanced Federated Fine-Tuning of Phi-4 Using OpenFL with PEFT & Quantization

In this tutorial, we demonstrate how to fine-tune Microsoft's Phi-4 model in a federated learning workflow with enhanced local training using:
- Parameter-Efficient Fine-Tuning (PEFT)
- 4-bit Quantization (QLoRA)
- Gradient Checkpointing
- Optimized Training Configuration

## Installation

In [ ]:
!pip install torch transformers peft datasets trl==0.12.2 bitsandbytes accelerate -q

## Import Libraries

In [ ]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict  # Added this import
from datasets import load_dataset
from trl import SFTTrainer
from openfl.experimental.workflow import FLSpec, Aggregator, Collaborator, LocalRuntime
import numpy as np

## Configuration

In [ ]:
# Model and dataset
model_name = "microsoft/phi-4"
dataset_name = "math_10k.json"

# QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# LoRA configuration
peft_config = LoraConfig(
    r=16,  # Increased from original for better adaptation
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],
)

# Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,  # Reduced for Phi-4
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none"
)

## Load and Prepare Model

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Load and Prepare Dataset

In [ ]:
def format_prompt(example):
    if example["input"]:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

# Load dataset
dataset = load_dataset("json", data_files=dataset_name, split="train")
dataset = dataset.map(lambda x: {"text": format_prompt(x)})

# Split dataset
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

## Enhanced Training with SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=True,
)

## Federated Averaging Function

In [ ]:
def FedAvg(peft_params, model, weights=None):
    """
    Perform Federated Averaging (FedAvg) on the model parameters.
    """
    state_dicts = peft_params
    state_dict = get_peft_model_state_dict(model)
    for key in peft_params[0]:
        dtype = state_dicts[0][key].dtype
        state_dict[key] = torch.from_numpy(
            np.average(
                [state[key].to(torch.float).numpy() for state in state_dicts], 
                axis=0, 
                weights=weights
            )
        ).to(dtype)
    set_peft_model_state_dict(model, state_dict)
    return model

## Federated Learning Workflow

In [ ]:
class FederatedFlow(FLSpec):
    def __init__(self, model=None, rounds=3, **kwargs):
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.peft_params = get_peft_model_state_dict(self.model)
        else:
            raise ValueError("No model provided")
        
        self.rounds = rounds
    
    @aggregator
    def start(self):
        print("Initializing federated learning")
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        self.next(self.aggregated_model_validation, foreach="collaborators")
    
    @collaborator
    def aggregated_model_validation(self):
        print(f"Validating aggregated model for {self.input}")
        # Load model with quantization
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, peft_config)
        set_peft_model_state_dict(self.model, self.peft_params)
        
        # Evaluate
        eval_results = trainer.evaluate()
        self.agg_validation_score = eval_results["eval_loss"]
        print(f"Validation loss: {self.agg_validation_score}")
        self.next(self.train)
    
    @collaborator
    def train(self):
        print(f"Training on {self.input}")
        # Train with local data
        trainer.train()
        self.loss = trainer.state.log_history[-1]["loss"]
        self.next(self.local_model_validation)
    
    @collaborator
    def local_model_validation(self):
        print(f"Validating local model for {self.input}")
        eval_results = trainer.evaluate()
        self.local_validation_score = eval_results["eval_loss"]
        self.peft_params = get_peft_model_state_dict(self.model)
        self.next(self.join, exclude=["model"])
    
    @aggregator
    def join(self, inputs):
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs
        ) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs
        ) / len(inputs)
        
        print(f"Round {self.current_round + 1} results:")
        print(f"Average training loss: {self.average_loss}")
        print(f"Average validation loss (before training): {self.aggregated_model_accuracy}")
        print(f"Average validation loss (after training): {self.local_model_accuracy}")
        
        # Federated averaging
        self.model = FedAvg([input.peft_params for input in inputs], self.model)
        self.peft_params = get_peft_model_state_dict(self.model)
        
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation, foreach="collaborators")
        else:
            self.next(self.end)
    
    @aggregator
    def end(self):
        print("Federated training complete!")
        print(f"Final model validation loss: {self.aggregated_model_accuracy}")

## Run Federated Learning

In [ ]:
# Setup participants
aggregator = Aggregator()
collaborators = [
    Collaborator(name="Portland"),
    Collaborator(name="Seattle"),
    Collaborator(name="London")
]

# Assign data shards
for idx, colab in enumerate(collaborators):
    colab.private_attributes = {
        "train_dataset": train_dataset.shard(len(collaborators), idx),
        "eval_dataset": eval_dataset.shard(len(collaborators), idx)
    }

# Create and run workflow
runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators)
flflow = FederatedFlow(model, rounds=3)
flflow.runtime = runtime
flflow.run()